In [ ]:
import sys
from pathlib import Path

import lightgbm as lgb
import numpy as np
import pandas as pd
from google.colab import drive
from sklearn.metrics import mean_absolute_error, mean_squared_error

# This works when the notebook is run from the cloned repository.
PROJECT_ROOT = Path.cwd()
for candidate in [PROJECT_ROOT, *PROJECT_ROOT.parents]:
    if (candidate / "models" / "tree_based" / "feature_engineering.py").exists():
        PROJECT_ROOT = candidate
        break

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from models.tree_based.feature_engineering import make_walmart_lgbm_feature_pipeline

drive.mount('/content/drive')


In [ ]:
DATA_DIR = Path("/content/drive/My Drive/walmart_competition_data")

df_train = pd.read_csv(DATA_DIR / "train.csv", parse_dates=["Date"])
df_test = pd.read_csv(DATA_DIR / "test.csv", parse_dates=["Date"])
df_features = pd.read_csv(DATA_DIR / "features.csv", parse_dates=["Date"])
df_stores = pd.read_csv(DATA_DIR / "stores.csv")


In [ ]:
def merge_walmart_tables(sales_frame, features, stores):
    frame = sales_frame.merge(stores, on="Store", how="left", validate="many_to_one")
    frame = frame.merge(
        features,
        on=["Store", "Date", "IsHoliday"],
        how="left",
        validate="many_to_one",
    )
    return frame.sort_values(["Date", "Store", "Dept"]).reset_index(drop=True)

train_merged = merge_walmart_tables(df_train, df_features, df_stores)
test_merged = merge_walmart_tables(df_test, df_features, df_stores)

train_merged.head()


In [ ]:
def weighted_mae(y_true, y_pred, is_holiday, holiday_weight=5.0):
    weights = np.where(np.asarray(is_holiday, dtype=bool), holiday_weight, 1.0)
    return float(np.average(np.abs(np.asarray(y_true) - np.asarray(y_pred)), weights=weights))

VALIDATION_WEEKS = 52
HOLIDAY_WEIGHT = 5.0

validation_start = train_merged["Date"].max() - pd.Timedelta(weeks=VALIDATION_WEEKS - 1)
train_mask = train_merged["Date"] < validation_start
valid_mask = ~train_mask

train_part = train_merged.loc[train_mask].copy()
valid_part = train_merged.loc[valid_mask].copy()

print("train:", train_part["Date"].min(), "to", train_part["Date"].max(), train_part.shape)
print("valid:", valid_part["Date"].min(), "to", valid_part["Date"].max(), valid_part.shape)


In [ ]:
feature_pipeline = make_walmart_lgbm_feature_pipeline(
    include_lag_features=True,
    drop_target_and_date=True,
)

X_train = feature_pipeline.fit_transform(train_part)
y_train = train_part["Weekly_Sales"]

X_valid = feature_pipeline.transform(valid_part)
y_valid = valid_part["Weekly_Sales"]

# LightGBM can use pandas category dtype directly.
categorical_features = X_train.select_dtypes(include="category").columns.tolist()

print("feature count:", X_train.shape[1])
print("categorical features:", categorical_features)
X_train.head()


In [ ]:
w_train = np.where(train_part["IsHoliday"].to_numpy(dtype=bool), HOLIDAY_WEIGHT, 1.0)
w_valid = np.where(valid_part["IsHoliday"].to_numpy(dtype=bool), HOLIDAY_WEIGHT, 1.0)

lgbm_model = lgb.LGBMRegressor(
    objective="regression_l1",
    n_estimators=3000,
    learning_rate=0.03,
    num_leaves=63,
    max_depth=-1,
    min_child_samples=50,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=0.1,
    random_state=42,
    n_jobs=-1,
)

lgbm_model.fit(
    X_train,
    y_train,
    sample_weight=w_train,
    eval_set=[(X_valid, y_valid)],
    eval_sample_weight=[w_valid],
    eval_metric="l1",
    categorical_feature=categorical_features,
    callbacks=[lgb.early_stopping(100), lgb.log_evaluation(100)],
)


In [ ]:
valid_pred = lgbm_model.predict(X_valid)
valid_holiday = valid_part["IsHoliday"].to_numpy(dtype=bool)

metrics = {
    "validation_wmae": weighted_mae(y_valid, valid_pred, valid_holiday, HOLIDAY_WEIGHT),
    "validation_mae": float(mean_absolute_error(y_valid, valid_pred)),
    "validation_rmse": float(mean_squared_error(y_valid, valid_pred) ** 0.5),
}

pd.Series(metrics, name="value").to_frame()


In [ ]:
feature_importance = (
    pd.DataFrame({
        "feature": X_train.columns,
        "importance": lgbm_model.feature_importances_,
    })
    .sort_values("importance", ascending=False)
    .reset_index(drop=True)
)

selected_features = feature_importance.loc[feature_importance["importance"] > 0, "feature"].tolist()
print("selected feature count:", len(selected_features), "of", X_train.shape[1])
feature_importance.head(30)


In [ ]:
# Optional second pass: retrain only on non-zero-importance features.
X_train_selected = X_train[selected_features]
X_valid_selected = X_valid[selected_features]
selected_categorical_features = [col for col in categorical_features if col in selected_features]

lgbm_selected_model = lgb.LGBMRegressor(
    objective="regression_l1",
    n_estimators=3000,
    learning_rate=0.03,
    num_leaves=63,
    max_depth=-1,
    min_child_samples=50,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=0.1,
    random_state=42,
    n_jobs=-1,
)

lgbm_selected_model.fit(
    X_train_selected,
    y_train,
    sample_weight=w_train,
    eval_set=[(X_valid_selected, y_valid)],
    eval_sample_weight=[w_valid],
    eval_metric="l1",
    categorical_feature=selected_categorical_features,
    callbacks=[lgb.early_stopping(100), lgb.log_evaluation(100)],
)

selected_valid_pred = lgbm_selected_model.predict(X_valid_selected)
selected_metrics = {
    "selected_validation_wmae": weighted_mae(y_valid, selected_valid_pred, valid_holiday, HOLIDAY_WEIGHT),
    "selected_validation_mae": float(mean_absolute_error(y_valid, selected_valid_pred)),
    "selected_validation_rmse": float(mean_squared_error(y_valid, selected_valid_pred) ** 0.5),
}

pd.Series({**metrics, **selected_metrics}, name="value").to_frame()
